In [1]:
# Core Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model Selection and Pipelines
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Preprocessing & Feature Engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer

# Algorithms (Built-in to sklearn)
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Evaluation Metrics (Regression focused)
from sklearn.metrics import (
    mean_absolute_error,    # MAE
    mean_squared_error,     # MSE
    root_mean_squared_error, # RMSE (Note: in newer sklearn versions)
    r2_score                # R-squared
)

# Utils
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
%matplotlib inline

In [2]:
df = pd.read_csv('../data/vehicles.csv')
df.sample(5)

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
248925,7312619361,https://lasvegas.craigslist.org/ctd/d/santa-cl...,las vegas,https://lasvegas.craigslist.org,11995,2007.0,NaN,HUMMER H3,NaN,5 cylinders,...,NaN,SUV,custom,https://images.craigslist.org/00l0l_iMQOFh5CC7...,This HUMMER H3 can be yours today!If you have ...,NaN,nv,37.175631,-113.60824,2021-04-25T18:01:46-0700
139027,7310406140,https://twinfalls.craigslist.org/ctd/d/las-veg...,twin falls,https://twinfalls.craigslist.org,39995,2016.0,chevrolet,silverado 2500hd work,NaN,NaN,...,NaN,NaN,NaN,https://images.craigslist.org/00a0a_1BqIhVrlie...,2016* *Chevrolet* *Silverado* *3500* *2500* *1...,NaN,id,36.114900,-115.21610,2021-04-21T11:05:33-0600
208543,7316830465,https://saginaw.craigslist.org/ctd/d/chesaning...,saginaw-midland-baycity,https://saginaw.craigslist.org,4995,2013.0,chrysler,200,NaN,NaN,...,NaN,NaN,NaN,https://images.craigslist.org/00101_2ETqmV5280...,WE HAVE OVER 350 VEHICLES IN STOCK! View Our...,NaN,mi,43.182400,-84.11220,2021-05-04T13:55:43-0400
371486,7314900953,https://dallas.craigslist.org/ftw/ctd/d/norman...,dallas / fort worth,https://dallas.craigslist.org,17988,2014.0,chevrolet,tahoe,NaN,8 cylinders,...,NaN,NaN,NaN,https://images.craigslist.org/00c0c_kKHJhRzSrx...,CHECK OUT THIS 2014 CHEVROLET TAHOE THAT WAS J...,NaN,tx,35.199000,-97.48410,2021-04-30T12:36:31-0500
10300,7316743109,https://phoenix.craigslist.org/nph/ctd/d/phoen...,phoenix,https://phoenix.craigslist.org,23590,2018.0,mazda,mazda6 touring sedan 4d,good,NaN,...,NaN,sedan,red,https://images.craigslist.org/00303_7loSHJn1r9...,Carvana is the safer way to buy a car During t...,NaN,az,33.500000,-112.05000,2021-05-04T08:41:05-0700


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   url           426880 non-null  object 
 2   region        426880 non-null  object 
 3   region_url    426880 non-null  object 
 4   price         426880 non-null  int64  
 5   year          425675 non-null  float64
 6   manufacturer  409234 non-null  object 
 7   model         421603 non-null  object 
 8   condition     252776 non-null  object 
 9   cylinders     249202 non-null  object 
 10  fuel          423867 non-null  object 
 11  odometer      422480 non-null  float64
 12  title_status  418638 non-null  object 
 13  transmission  424324 non-null  object 
 14  VIN           265838 non-null  object 
 15  drive         296313 non-null  object 
 16  size          120519 non-null  object 
 17  type          334022 non-null  object 
 18  pain

In [4]:
# id column is not useful and it is unique for each row
# url, region_url, image_url, description are not useful, description is unique long text for each row
#VIN is unique for each car, and it has a lot of missing values, so it is not useful 
# county has 0 non-null values, so it is not useful
# dropping these immidiately
initial_drop_cols = ['id','url', 'region_url', 'image_url', 'description', 'county', 'VIN']
df.drop(columns=initial_drop_cols, inplace=True)

In [5]:
# copying the dataframe to have a backup of the original data
df_copy = df.copy()

In [6]:
df['region'].value_counts()

region
columbus                   3608
jacksonville               3562
spokane / coeur d'alene    2988
eugene                     2985
fresno / madera            2983
                           ... 
meridian                     28
southwest MS                 14
kansas city                  11
fort smith, AR                9
west virginia (old)           8
Name: count, Length: 404, dtype: int64

In [7]:
# region is categorical and has 404 unique values, too much for one hot encoding, so dropping
df.drop(columns=['region'], inplace=True)

In [8]:
print(df['state'].nunique())
df['state'].value_counts()

51


state
ca    50614
fl    28511
tx    22945
ny    19386
oh    17696
or    17104
mi    16900
nc    15277
wa    13861
pa    13753
wi    11398
co    11088
tn    11066
va    10732
il    10387
nj     9742
id     8961
az     8679
ia     8632
ma     8174
mn     7716
ga     7003
ok     6792
sc     6327
mt     6294
ks     6209
in     5704
ct     5188
al     4955
md     4778
nm     4425
mo     4293
ky     4149
ar     4038
ak     3474
la     3196
nv     3194
nh     2981
dc     2970
me     2966
hi     2964
vt     2513
ri     2320
sd     1302
ut     1150
wv     1052
ne     1036
ms     1016
de      949
wy      610
nd      410
Name: count, dtype: int64

In [ ]:
# state has 51 unique values, but we leave it for now, decide later

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   price         426880 non-null  int64  
 1   year          425675 non-null  float64
 2   manufacturer  409234 non-null  object 
 3   model         421603 non-null  object 
 4   condition     252776 non-null  object 
 5   cylinders     249202 non-null  object 
 6   fuel          423867 non-null  object 
 7   odometer      422480 non-null  float64
 8   title_status  418638 non-null  object 
 9   transmission  424324 non-null  object 
 10  drive         296313 non-null  object 
 11  size          120519 non-null  object 
 12  type          334022 non-null  object 
 13  paint_color   296677 non-null  object 
 14  state         426880 non-null  object 
 15  lat           420331 non-null  float64
 16  long          420331 non-null  float64
 17  posting_date  426812 non-null  object 
dtypes: f

In [10]:
df.sample(5)

,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,lat,long,posting_date
112817,2500,2005.0,chrysler,sebring convertible,excellent,NaN,gas,156000.0,clean,automatic,NaN,NaN,NaN,NaN,fl,25.989400,-80.215300,2021-05-01T07:46:10-0400
10744,14991,2015.0,honda,civic sedan lx,good,4 cylinders,gas,70544.0,clean,automatic,fwd,NaN,sedan,black,az,33.416084,-111.791486,2021-05-03T16:51:03-0700
309716,7575,2009.0,subaru,outback 2.5i limited,excellent,NaN,gas,124588.0,clean,automatic,NaN,NaN,NaN,NaN,ok,35.990872,-95.886376,2021-04-17T10:29:39-0500
307732,0,2014.0,ford,f-250sd,excellent,8 cylinders,diesel,138319.0,clean,automatic,4wd,full-size,truck,brown,ok,34.197090,-97.160560,2021-04-10T09:26:48-0500
194565,15800,2000.0,pontiac,firebird transam,excellent,8 cylinders,gas,56000.0,clean,automatic,rwd,full-size,convertible,red,mi,42.358300,-83.900000,2021-04-08T11:02:43-0400


In [11]:
# lats and longs: bigger numbers dont necessarily mean more expensive cars, so we can drop them for now, decide later if we want to use them or not
# also have about 8000 missing values
df.drop(columns=['lat', 'long'], inplace=True)

In [12]:
# making the number of cylinders an integer by removing the 'cylinders' string and converting to numeric
df['cylinders'] = df['cylinders'].str.replace('cylinders', '').str.strip()
df['cylinders'] = pd.to_numeric(df['cylinders'], errors='coerce')


In [13]:
df.describe()

,price,year,cylinders,odometer
count,4.268800e+05,425675.000000,247904.000000,4.224800e+05
mean,7.519903e+04,2011.235191,5.968685,9.804333e+04
std,1.218228e+07,9.452120,1.602962,2.138815e+05
min,0.000000e+00,1900.000000,3.000000,0.000000e+00
25%,5.900000e+03,2008.000000,4.000000,3.770400e+04
50%,1.395000e+04,2013.000000,6.000000,8.554800e+04
75%,2.648575e+04,2017.000000,8.000000,1.335425e+05
max,3.736929e+09,2022.000000,12.000000,1.000000e+07


In [ ]:
# loaded the data, 426880 rows
# condition will be filled with 'not_reported' because it has almost 45  percent missing values, and it is a categorical variable.
# cylinders will be filled with the most frequent value for that particular car brand/model. so group by model and take mode
# will use functiontransformer in my pipeline for cylinder and age to fill the missing values.
# will probably reduce the number of states by grouping them into regions, because there are 51 unique values and it is a categorical variable., about 10 regions before one hot encoding.
# will drop the original region column cuz too many unique values and it is a categorical variable.
# will calculate the age of the car using a function transformer and drop the year posted column which i will get from the date posted column.
# will input the mileage with the median/mean value based on model/brand and the age
# might add poly features for age and mileage
# will leave the price at 500k and log transform it to balance the model's attention


In [14]:
# the target range is unreasoble, between 0 and 3.7 billion dollars. filter it to 500 to 500k
df_unfiltered =df.copy()   # keeping a copy of the unfiltered data for later use if needed
df= df[(df['price']<500000) & (df['price']>500)]
print(df.shape)

(383697, 16)


In [16]:
# the posted date column has month and time which are not useful because there is no other detail time column to compare with, will extract just the year
df['year_posted'] = pd.to_datetime(df['posting_date'], errors='coerce', utc=True).dt.year
df.drop(columns=['posting_date'], inplace=True)
df.sample(5)

,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state,year_posted
96001,5988,2011.0,chrysler,300,excellent,NaN,gas,123545.0,clean,automatic,rwd,NaN,sedan,grey,fl,2021.0
257367,8248,2008.0,ford,edge,excellent,6.0,gas,79572.0,clean,automatic,fwd,NaN,sedan,silver,nj,2021.0
275722,8995,2008.0,honda,cr-v,excellent,NaN,gas,102395.0,clean,automatic,4wd,NaN,SUV,grey,ny,2021.0
186279,22590,2016.0,subaru,crosstrek 2.0i limited,good,NaN,other,28379.0,clean,automatic,NaN,NaN,hatchback,NaN,ma,2021.0
5977,17995,2015.0,chevrolet,impala lt v6,like new,6.0,gas,75369.0,clean,automatic,fwd,full-size,sedan,white,ak,2021.0


In [18]:
df['year_posted'] = df['year_posted'].astype('Int64')
df['year_posted'].value_counts().sort_index(ascending=False).head()

year_posted
2021    383636
Name: count, dtype: Int64

** Okay this is basically useless as a feature since it has only 1 unique value, so we will focus on timedelta ie the car's age instead of absolute time: drop both manufactured year and year posted after calculating age
- It has about 60 nans, cannot calculate age with those, so dropping it

In [19]:
# dropping nans from year_posted
df.dropna(subset=['year_posted'], inplace=True)

** Now to our target column. The statistics for it did not look very good, 
- I already dropped outrageous values for price, Now the new distribution

In [20]:
# Percentile check
print(df['price'].quantile([0.9, 0.95, 0.99, 0.999]))

0.900     38590.00
0.950     45500.00
0.990     68788.00
0.999    119641.62
Name: price, dtype: float64


In [22]:
df['price'].isnull().sum()

np.int64(0)

** Therefore we still have 99.9 percent of our data despite dropping entries with cars below 500 dollars and above 500k dollars. 
** No nan values in the target column good

In [21]:
print('high end values:', (df['price']>=300000).sum())
print('luxury cars:', ((df['price']>=100000) & (df['price']<300000)).sum())

high end values: 9
luxury cars: 620


** Handling the year column
- The age will be a very useful column so we must know when a car was manufactured, therefore dropping nan year rows
- Also dropping any entry where the manufactured date is above 2021 because the posted date is 2021 and these are supposed to be already used cars, cannt have a negative age

In [23]:
# cleaning/handling the year column
df['year'] = df['year'].astype('Int64') 
df.dropna(subset=['year'], inplace=True)
print(df['year'].value_counts().sort_index(ascending=False).head(20))

year
2022       43
2021     1645
2020    17490
2019    22318
2018    31484
2017    31306
2016    26580
2015    27058
2014    26160
2013    27422
2012    21790
2011    18586
2010    14626
2009    11447
2008    16073
2007    13970
2006    11975
2005    10105
2004     8549
2003     6830
Name: count, dtype: Int64


In [24]:
# checking how many cars are older than 1960, which is a reasonable cutoff for used cars, and we can drop the older ones
(df['year']<1960).sum()

np.int64(2298)

In [25]:
# filtering out cars with manufacturing year before 1960 and after 2021
df = df[(df['year'] >= 1960) & (df['year'] <= 2021)]